In [0]:
from datetime import date
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

In [0]:

features = spark.table("workspace.mortgage_risk.gold_mortgage_features")
features.printSchema()

display(features.agg(F.count("*").alias("feature_rows"),
        F.max("prediction_month").alias("latest_available_month")))

root
 |-- loan_id: string (nullable = true)
 |-- prediction_month: date (nullable = true)
 |-- current_upb: decimal(18,2) (nullable = true)
 |-- current_rate: double (nullable = true)
 |-- estimated_ltv: double (nullable = true)
 |-- loan_age: integer (nullable = true)
 |-- remaining_term: integer (nullable = true)
 |-- delinquency_months: integer (nullable = true)
 |-- delinquency_unknown: integer (nullable = true)
 |-- history_months_6m: long (nullable = true)
 |-- unknown_months_6m: long (nullable = true)
 |-- late_months_3m: long (nullable = true)
 |-- late_months_6m: long (nullable = true)
 |-- max_delinquency_3m: integer (nullable = true)
 |-- max_delinquency_6m: integer (nullable = true)
 |-- first_payment_date: date (nullable = true)
 |-- fico: integer (nullable = true)
 |-- ltv: double (nullable = true)
 |-- cltv: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- original_upb: decimal(18,2) (nullable = true)
 |-- original_rate: double (nullable = true)
 |-- orig

feature_rows,latest_available_month
16407101,2026-03-01


In [0]:
dbutils.widgets.text("prediction_month", "", "Feature month (YYYY-MM-01)")

requested_month = dbutils.widgets.get("prediction_month").strip()

model_folder = "/Volumes/workspace/mortgage_risk/model_artifacts"

model_version = "gbt_v1"
calibration_version = "gbt_v1_calibration_v1"
alert_fraction = 0.05

score_table = "workspace.mortgage_risk.monthly_risk_scores"
run_id = str(uuid.uuid4())

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5364605774733530>, line 12
      9 alert_fraction = 0.05
     11 score_table = "workspace.mortgage_risk.monthly_risk_scores"
---> 12 run_id = str(uuid.uuid4())

NameError: name 'uuid' is not defined

In [0]:
features = spark.table("workspace.mortgage_risk.gold_mortgage_features")
if requested_month:
    scoring_month = date.fromisoformat(requested_month)
    assert scoring_month.day == 1, "Use the first day of the month."
else:
    scoring_month = features.agg(F.max("prediction_month")).first()[0]
    assert scoring_month is not None, "The feature table is empty."

monthly_input = features.filter(
    F.col("prediction_month") == F.lit(scoring_month))
input_rows = monthly_input.count()

assert input_rows > 0, "No eligible observations for this month."

print("Feature month:", scoring_month)
print("Eligible observations:", input_rows)

Feature month: 2026-03-01
Eligible observations: 258474


In [0]:
# Select the month to score
monthly_input = monthly_input.withColumn(
    "original_age_months",
    F.months_between(
        "prediction_month", "first_payment_date"
    ).cast("int") + 1
)

monthly_input = monthly_input.withColumn(
    "multiple_borrowers",
    F.when(F.col("borrowers") > 1, 1)
     .when(F.col("borrowers") == 1, 0)
     .otherwise(F.lit(None))
)

monthly_input = monthly_input.withColumn(
    "rate_change",
    F.col("current_rate") - F.col("original_rate")
)

In [0]:
# Recreate the three derived features
monthly_input = monthly_input.withColumn(
    "original_age_months",
    F.months_between(
        "prediction_month", "first_payment_date"
    ).cast("int") + 1
)

monthly_input = monthly_input.withColumn(
    "multiple_borrowers",
    F.when(F.col("borrowers") > 1, 1)
     .when(F.col("borrowers") == 1, 0)
     .otherwise(F.lit(None))
)

monthly_input = monthly_input.withColumn(
    "rate_change",
    F.col("current_rate") - F.col("original_rate")
)

In [0]:
# Match the training input types
numeric_columns = [
    "fico", "ltv", "additional_lien_ltv", "dti",
    "original_upb", "original_rate", "original_term", "mi_pct",
    "multiple_borrowers", "units", "original_age_months",
    "loan_age", "remaining_term", "current_rate", "rate_change",
    "estimated_ltv", "balance_ratio", "delinquency_months",
    "late_months_3m", "late_months_6m",
    "max_delinquency_3m", "max_delinquency_6m"
]

categorical_columns = [
    "purpose", "occupancy", "property_type", "state", "channel"
]

required_columns = numeric_columns + categorical_columns + [
    "loan_id", "prediction_month", "source_vintage"
]

missing_columns = []

for column in required_columns:
    if column not in monthly_input.columns:
        missing_columns.append(column)

assert not missing_columns, f"Missing model inputs: {missing_columns}"

for column in numeric_columns:
    monthly_input = monthly_input.withColumn(
        column, F.col(column).cast("double")
    )

print("All model inputs are ready.")

All model inputs are ready.


In [0]:
# Check the loan-month identifiers
missing_ids = monthly_input.filter(
    F.col("loan_id").isNull()
    | (F.trim("loan_id") == "")
).count()

assert missing_ids == 0, "Missing loan identifiers."

duplicate_ids = monthly_input.groupBy("loan_id", "prediction_month").count()
duplicate_ids = duplicate_ids.filter(F.col("count") > 1).limit(1).count()

assert duplicate_ids == 0, "Duplicate loan-month observations."

print("Loan-month identifier checks passed.")

Loan-month identifier checks passed.


In [0]:
# Load the saved models
gbt_model = PipelineModel.load(model_folder + "/gradient_boosted_trees_v1")
calibration_model = PipelineModel.load(model_folder + "/gbt_v1_calibration_v1")

assert calibration_model.stages[-1].coefficients[0] > 0


In [0]:
# Generate the monthly scores
scored = gbt_model.transform(monthly_input)

scored = scored.withColumn("risk_score",
    vector_to_array("probability")[1]
)

scored = calibration_model.transform(scored)

scored = scored.withColumn(
    "calibrated_risk",
    vector_to_array("calibrated_probability")[1]
).select(
    "loan_id", "prediction_month", "source_vintage",
    "state", "purpose", "occupancy", "property_type",
    "fico", "ltv", "estimated_ltv", "original_upb", "balance_ratio",
    "delinquency_months", "risk_score", "calibrated_risk"
)

display(scored.limit(5))

loan_id,prediction_month,source_vintage,state,purpose,occupancy,property_type,fico,ltv,estimated_ltv,original_upb,balance_ratio,delinquency_months,risk_score,calibrated_risk
F16Q10168268,2026-03-01,2016,WA,N,P,PU,706.0,75.0,30.0,541000.0,0.7832026247689464,0.0,0.038881579366333074,0.005009413724032075
F16Q10280209,2026-03-01,2016,CA,N,P,SF,789.0,48.0,18.0,165000.0,0.7085540606060606,0.0,0.02417545720858172,0.002459571487478218
F16Q10280536,2026-03-01,2016,CA,N,P,SF,738.0,58.0,25.0,227000.0,0.7830969603524229,0.0,0.027481602628340318,0.0029772948611952454
F16Q20051076,2026-03-01,2016,OH,P,P,CO,764.0,90.0,34.0,113000.0,0.7874806194690265,0.0,0.030043760474305237,0.00340144399229525
F16Q20129074,2026-03-01,2016,CA,P,I,SF,716.0,85.0,38.0,146000.0,0.8139850684931508,0.0,0.03878930112519352,0.004991507007201301


In [0]:
# Make sure scoring kept every input row.
scored_rows = scored.count()

print("Input rows:", input_rows)
print("Scored rows:", scored_rows)

assert scored_rows == input_rows, "Some rows were lost during scoring."

# Check that both scores are present and between 0 and 1.
for column in ["risk_score", "calibrated_risk"]:
    invalid_rows = scored.filter(
        F.col(column).isNull()
        | F.isnan(column)
        | (F.col(column) < 0)
        | (F.col(column) > 1)
    ).count()

    print(column, "- invalid rows:", invalid_rows)

    assert invalid_rows == 0, f"Check the values in {column}."

healthy_rows = scored.filter(
    F.col("delinquency_months") == 0
).count()

print("Current-status-00 rows:", healthy_rows)
print("Prediction checks passed.")

Input rows: 258474
Scored rows: 258474
risk_score - invalid rows: 0
calibrated_risk - invalid rows: 0
Current-status-00 rows: 256091
Prediction checks passed.


In [0]:
# Review all eligible loans.
all_loans = scored.withColumn(
    "population", F.lit("All eligible loans"))

# Also review loans that are currently up to date.
current_loans = scored.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population", F.lit("Current status 00"))

review_scores = all_loans.unionByName(current_loans)

In [0]:
# Rank loans and flag the top 5%
alert_fraction = 0.05

month_window = Window.partitionBy(
    "population", "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

monthly_scores = review_scores.withColumn(
    "monthly_rows", F.count("*").over(month_window)
)

monthly_scores = monthly_scores.withColumn(
    "risk_rank", F.row_number().over(risk_window)
)

monthly_scores = monthly_scores.withColumn(
    "alert_limit",
    F.ceil(F.col("monthly_rows") * alert_fraction).cast("long")
)

monthly_scores = monthly_scores.withColumn(
    "alert",
    F.when(F.col("risk_rank") <= F.col("alert_limit"), 1)
     .otherwise(0)
)

In [0]:
# Record which model and run produced the scores
import uuid

model_version = "gbt_v1"
calibration_version = "gbt_v1_calibration_v1"
score_table = "workspace.mortgage_risk.monthly_risk_scores"

run_id = str(uuid.uuid4())

monthly_scores = monthly_scores.withColumn(
    "model_version", F.lit(model_version))

monthly_scores = monthly_scores.withColumn(
    "calibration_version", F.lit(calibration_version))

monthly_scores = monthly_scores.withColumn(
    "alert_fraction", F.lit(alert_fraction))

monthly_scores = monthly_scores.withColumn(
    "run_id", F.lit(run_id))

monthly_scores = monthly_scores.withColumn(
    "scored_at", F.current_timestamp())

In [0]:
# Save the monthly results
assert input_rows > 0, "There are no observations to save."

replace_condition = (
    f"prediction_month = DATE '{scoring_month.isoformat()}' "
    f"AND model_version = '{model_version}' "
    f"AND calibration_version = '{calibration_version}' "
    f"AND alert_fraction = {alert_fraction}"
)

if not spark.catalog.tableExists(score_table):
    monthly_scores.write.format("delta").mode("error").partitionBy(
        "prediction_month"
    ).saveAsTable(score_table)

else:
    monthly_scores.write.format("delta").mode("overwrite").option(
        "replaceWhere", replace_condition
    ).saveAsTable(score_table)

saved_scores = spark.table(score_table).filter(replace_condition)

print("Saved monthly scores for:", scoring_month)

Saved monthly scores for: 2026-03-01


In [0]:
# Check the saved records and alert counts
saved_all_rows = saved_scores.filter(
    F.col("population") == "All eligible loans"
).count()

saved_current_rows = saved_scores.filter(
    F.col("population") == "Current status 00"
).count()

assert saved_all_rows == input_rows, "All-loan row count does not match."
assert saved_current_rows == healthy_rows, "Current-loan row count does not match."

output_keys = [
    "loan_id",
    "prediction_month",
    "population",
    "model_version",
    "calibration_version",
    "alert_fraction"
]

duplicates = saved_scores.groupBy(*output_keys).count()

assert duplicates.filter(
    F.col("count") > 1
).limit(1).count() == 0, "Duplicate saved records found."

monthly_summary = saved_scores.groupBy(
    "population", "prediction_month"
).agg(
    F.count("*").alias("eligible_observations"),
    F.sum("alert").alias("alerts"),
    F.max("alert_limit").alias("expected_alerts"),
    F.avg("calibrated_risk").alias("average_estimated_risk")
)

wrong_alert_counts = monthly_summary.filter(
    F.col("alerts") != F.col("expected_alerts")
).count()

assert wrong_alert_counts == 0, "Alert counts do not match the budget."

display(monthly_summary.orderBy("population"))

population,prediction_month,eligible_observations,alerts,expected_alerts,average_estimated_risk
All eligible loans,2026-03-01,258474,12924,12924,0.008567697012952579
Current status 00,2026-03-01,256091,12805,12805,0.005175417674990471


In [0]:
# View the highest-risk loans that are currently up to date
current_alerts = saved_scores.filter(
    (F.col("population") == "Current status 00")
    & (F.col("alert") == 1)
)

display(
    current_alerts.select(
        "loan_id",
        "prediction_month",
        "risk_rank",
        "state",
        "fico",
        "ltv",
        "estimated_ltv",
        "risk_score",
        "calibrated_risk"
    ).orderBy("risk_rank").limit(20)
)

loan_id,prediction_month,risk_rank,state,fico,ltv,estimated_ltv,risk_score,calibrated_risk
F23Q10092122,2026-03-01,1,TX,694.0,95.0,84.0,0.5062966368763882,0.35776795615091994
F22Q20316409,2026-03-01,2,FL,677.0,90.0,94.0,0.49161624744563726,0.3383714443056747
F23Q10030724,2026-03-01,3,CA,704.0,95.0,80.0,0.4749651069330081,0.316989349213589
F24Q30082749,2026-03-01,4,NY,641.0,91.0,79.0,0.47021515297101435,0.3110132919506371
F23Q20128680,2026-03-01,5,MO,715.0,95.0,87.0,0.46894871628029233,0.30942932994498296
F22Q40109250,2026-03-01,6,WI,698.0,95.0,80.0,0.45344650945627907,0.29036392199914884
F23Q20127740,2026-03-01,7,MI,633.0,90.0,84.0,0.4477032696697636,0.2834540551352841
F24Q20086459,2026-03-01,8,KY,659.0,90.0,85.0,0.444163165595875,0.27923660634078074
F23Q40166979,2026-03-01,9,GA,673.0,90.0,89.0,0.4422373989345001,0.27695580338750025
F23Q30098161,2026-03-01,10,NY,721.0,92.0,69.0,0.4391282604918816,0.2732934921873509


In [0]:
display(monthly_summary.orderBy("population"))

population,prediction_month,eligible_observations,alerts,expected_alerts,average_estimated_risk
All eligible loans,2026-03-01,258474,12924,12924,0.008567697012952579
Current status 00,2026-03-01,256091,12805,12805,0.005175417674990471
